In [1]:
import os
import pandas as pd
import numpy as np
#from libtiff import TIFF
from matplotlib import pyplot as plt
from skimage.segmentation import find_boundaries
import seaborn as sns
import colorcet as cc
import tifffile
import scipy as sc

### directly get through the all excel by pandas

In [2]:
path = '../data/RNA_meta_final.csv'
#Mind that the data we have here are 
df_all_anno = pd.read_csv(path)
df_all_anno.annV7_noNum.value_counts()

/var/folders/3_/pf5ff_p57fqfsm6w00s_hg480000gn/T/ipykernel_86784/2412377179.py:7: DtypeWarning: Columns (30,31,32,35,36,38,39,40,41,43,44,46,48,49,50,51,54,55,59,66,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,90,91,92,94,95,96,97,98,100,104,105,106,107,108,109,110,111,113) have mixed types. Specify dtype option on import or set low_memory=False.
  df_all_anno = pd.read_csv(path)


annV7_noNum
Carcinoma                   84037
Sarcoma                     75625
Ambiguous                   20351
Benign Stroma               17537
Macrophage/DC               16603
Myometrium                  10642
Endothelial                 10251
Plasma/B                     6813
T                            6296
Benign Endometrial gland     1056
Undif Carci                   759
Name: count, dtype: int64

In [11]:
# use this to avoid unique panel for each FOV
df_all_anno['annV7_noNum'] = df_all_anno['annV7_noNum'].astype('category')

## set up color panel
class_names = [
                  "Ambiguous",
                  "Benign Endometrial gland",
                  "Benign Stroma",
                  "Carcinoma",
                  "Endothelial",
                  "Macrophage/DC",
                  "Myometrium",
                  "Plasma/B",
                  "Sarcoma",
                  "T",
                  "Undif Carci"]
# make color panel with more than 20 colors
class_colors = sns.color_palette(['#4363d8',
'#800000',
'#3cb44b',
'#000075',
'#ffe119',
'#46f0f0',
'#911eb4',
'#e6beff',
'#ffd8b1',
'#008080',
'#000000'])
class_colors

[(0.2627450980392157, 0.38823529411764707, 0.8470588235294118),
 (0.5019607843137255, 0.0, 0.0),
 (0.23529411764705882, 0.7058823529411765, 0.29411764705882354),
 (0.0, 0.0, 0.4588235294117647),
 (1.0, 0.8823529411764706, 0.09803921568627451),
 (0.27450980392156865, 0.9411764705882353, 0.9411764705882353),
 (0.5686274509803921, 0.11764705882352941, 0.7058823529411765),
 (0.9019607843137255, 0.7450980392156863, 1.0),
 (1.0, 0.8470588235294118, 0.6941176470588235),
 (0.0, 0.5019607843137255, 0.5019607843137255),
 (0.0, 0.0, 0.0)]

In [12]:

fromPath = '../../../../data_from_YH/'

for fileName in df_all_anno['tma_fov'].unique():
    # plot specific FOV
    tempDf = df_all_anno[df_all_anno['tma_fov'] == fileName]
    print('Generating prediction maps for image: %s' % fileName)
    
    # load cell boundaries
    currTMA = fileName.split('_')[0]
    currFOV = fileName.split('_')[-1]
    subpath_u = '/TMA'+currTMA+'_segmentation/'

    if currTMA == '4':
        subpath = '20221005_191422_S1_C902_P99_N99_'
    if currTMA == '6':
        subpath = '20221003_185833_S2_C902_P99_N99_'
    if currTMA == '7':
        subpath = '20221005_191422_S3_C902_P99_N99_'

    if int(currFOV) < 10:
        cellBoundary_path = fromPath + subpath_u + subpath+'F00'+currFOV+'/0.25mpp_0.075maxima_0.01interior/seg_outline.tiff'
        cell_ins_map_path = fromPath + subpath_u + subpath+'F00'+currFOV+'/0.25mpp_0.075maxima_0.01interior/MESMER_mask.tiff'
        #cellBoundary
    else:
        cellBoundary_path = fromPath + subpath_u + subpath+'F0'+currFOV+'/0.25mpp_0.075maxima_0.01interior/seg_outline.tiff'
        cell_ins_map_path = fromPath + subpath_u + subpath+'F0'+currFOV+'/0.25mpp_0.075maxima_0.01interior/MESMER_mask.tiff'
    # get the cell boundaries
    cellBoundary = tifffile.imread(cellBoundary_path)
    cellBoundary.astype('uint8')
    # dilate the boundary to make the boundaries more obvious
    struct1 = sc.ndimage.generate_binary_structure(2, 1)
    cellBoundary = sc.ndimage.binary_dilation(cellBoundary, structure=struct1,iterations=3).astype(cellBoundary.dtype)
    
    # get the seg
    cell_ins_map = tifffile.imread(cell_ins_map_path)

    r = np.zeros((cellBoundary.shape[0], cellBoundary.shape[1]))
    g = np.zeros((cellBoundary.shape[0], cellBoundary.shape[1]))
    b = np.zeros((cellBoundary.shape[0], cellBoundary.shape[1]))

    cell_count = tempDf.shape[0]
    for i, row in tempDf.iterrows():
        print('Cell %d/%d - %s         ' % (int(row['cellLabel']), cell_count, fileName), end='\r')
        
        cell_id = int(row['cellLabel'])
        cell_pred = row['annV7_noNum']
        
        mask = cell_ins_map == cell_id
        r[mask] = class_colors[class_names.index(cell_pred)][0]
        g[mask] = class_colors[class_names.index(cell_pred)][1]
        b[mask] = class_colors[class_names.index(cell_pred)][2]

    print('')
    rgb = np.stack([r, g, b], axis=-1)
    # draw the boundaries
    rgb[:,:,0] = rgb[:,:,0]*255 + 40*cellBoundary
    rgb[:,:,1] = rgb[:,:,1]*255 + 40*cellBoundary
    rgb[:,:,2] = rgb[:,:,2]*255 + 40*cellBoundary

    rgb[rgb>255] = 255
    
    # define results dir
    resultDir = os.path.join('../plot/rna_pheno_low/')
    plt.imsave(os.path.join(resultDir, fileName+'.png'), np.uint8(rgb))

Generating prediction maps for image: 4_2
Cell 4901/4347 - 4_2         
Generating prediction maps for image: 4_3
Cell 3857/3573 - 4_3         
Generating prediction maps for image: 4_4
Cell 5514/5118 - 4_4         
Generating prediction maps for image: 4_5
Cell 1993/1872 - 4_5         
Generating prediction maps for image: 4_6
Cell 2928/2319 - 4_6         
Generating prediction maps for image: 4_7
Cell 6558/6298 - 4_7         
Generating prediction maps for image: 4_8
Cell 5073/3712 - 4_8         
Generating prediction maps for image: 4_9
Cell 5422/5272 - 4_9         
Generating prediction maps for image: 4_10
Cell 4550/4390 - 4_10         
Generating prediction maps for image: 4_11
Cell 5603/5388 - 4_11         
Generating prediction maps for image: 4_12
Cell 4440/3907 - 4_12         
Generating prediction maps for image: 4_14
Cell 2127/2004 - 4_14         
Generating prediction maps for image: 4_15
Cell 6829/5474 - 4_15         
Generating prediction maps for image: 4_16
Cell 3319/3